In [1]:
# Install poppler-utils for PDF rendering on Linux
!apt-get update -y && !apt-get install -y poppler-utils

# Install python dependencies
!pip install pdf2image easyocr opencv-python-headless numpy

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]      
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease                         
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]           
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]        
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,318 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,040 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]     
Get:1

In [3]:
!apt-get update -y && !apt-get install -y poppler-utils

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease                         
Hit:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease                 
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease               
Hit:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease                     
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease   
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION se

In [4]:
import os
import cv2
import numpy as np
import easyocr
from pdf2image import convert_from_path

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
pdf_path = "/kaggle/input/datasets/huseelbeera/gazette/----.pdf"
output_file = "/kaggle/working/easyocr_extracted_arabic.txt"
CONFIDENCE_THRESHOLD = 0.40

# ---------------------------------------------------------
# Initialize GPU EasyOCR Reader
# ---------------------------------------------------------
print("Initializing EasyOCR Engine on GPU...")
reader = easyocr.Reader(['ar'], gpu=True)

# ---------------------------------------------------------
# Convert PDF to Images
# ---------------------------------------------------------
print(f"Loading PDF from: {pdf_path}")
pages = convert_from_path(pdf_path, dpi=300)
print(f"Total Pages to Process: {len(pages)}")

# ---------------------------------------------------------
# Batch OCR Processing
# ---------------------------------------------------------
extracted_pages = []

for page_idx, page in enumerate(pages, 1):
    print(f"Processing Page {page_idx}/{len(pages)} on GPU...")
    img_np = np.array(page)
    
    # Preprocessing pipeline
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    enhanced = cv2.convertScaleAbs(gray, alpha=1.5, beta=-50)
    clean_img = cv2.medianBlur(enhanced, 3)
    
    # GPU Text extraction
    results = reader.readtext(clean_img, detail=1)
    
    # Sort top-to-bottom based on Y-coordinate
    results.sort(key=lambda x: x[0][0][1])
    
    page_lines = []
    for bbox, text, score in results:
        if score >= CONFIDENCE_THRESHOLD and text.strip():
            page_lines.append(text.strip())
            
    # Format page text
    page_content = f"--- PAGE {page_idx} ---\n" + "\n".join(page_lines)
    extracted_pages.append(page_content)

# ---------------------------------------------------------
# Save File to Kaggle Working Directory
# ---------------------------------------------------------
full_document_text = "\n\n".join(extracted_pages)
with open(output_file, "w", encoding="utf-8") as f:
    f.write(full_document_text)

print(f"\nExtraction complete! Saved to: {output_file}")

Initializing EasyOCR Engine on GPU...
Loading PDF from: /kaggle/input/datasets/huseelbeera/gazette/----.pdf


PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

In [5]:
!pip install pymupdf easyocr opencv-python-headless numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 76.5 MB/s eta 0:00:00:00:0100:01


In [6]:
import os
import cv2
import fitz  # PyMuPDF
import numpy as np
import easyocr

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
pdf_path = "/kaggle/input/datasets/huseelbeera/gazette/----.pdf"
output_file = "/kaggle/working/easyocr_extracted_arabic.txt"
CONFIDENCE_THRESHOLD = 0.40

# ---------------------------------------------------------
# Initialize GPU EasyOCR Reader
# ---------------------------------------------------------
print("Initializing EasyOCR Engine on GPU...")
reader = easyocr.Reader(['ar'], gpu=True)

# ---------------------------------------------------------
# Load PDF via PyMuPDF (No Poppler required)
# ---------------------------------------------------------
print(f"Loading PDF from: {pdf_path}")
doc = fitz.open(pdf_path)
total_pages = len(doc)
print(f"Total Pages to Process: {total_pages}")

# ---------------------------------------------------------
# Batch OCR Processing
# ---------------------------------------------------------
extracted_pages = []

for page_idx in range(total_pages):
    print(f"Processing Page {page_idx + 1}/{total_pages} on GPU...")
    
    # Render PDF page to RGB image array at 300 DPI
    page = doc[page_idx]
    pix = page.get_pixmap(dpi=300)
    img_np = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    
    # Preprocessing pipeline
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    enhanced = cv2.convertScaleAbs(gray, alpha=1.5, beta=-50)
    clean_img = cv2.medianBlur(enhanced, 3)
    
    # GPU Text extraction
    results = reader.readtext(clean_img, detail=1)
    
    # Sort top-to-bottom based on Y-coordinate
    results.sort(key=lambda x: x[0][0][1])
    
    page_lines = []
    for bbox, text, score in results:
        if score >= CONFIDENCE_THRESHOLD and text.strip():
            page_lines.append(text.strip())
            
    # Format page text
    page_content = f"--- PAGE {page_idx + 1} ---\n" + "\n".join(page_lines)
    extracted_pages.append(page_content)

# ---------------------------------------------------------
# Save File to Kaggle Working Directory
# ---------------------------------------------------------
full_document_text = "\n\n".join(extracted_pages)
with open(output_file, "w", encoding="utf-8") as f:
    f.write(full_document_text)

print(f"\nExtraction complete! Saved to: {output_file}")

Initializing EasyOCR Engine on GPU...
Loading PDF from: /kaggle/input/datasets/huseelbeera/gazette/----.pdf
Total Pages to Process: 58
Processing Page 1/58 on GPU...
Processing Page 2/58 on GPU...
Processing Page 3/58 on GPU...
Processing Page 4/58 on GPU...
Processing Page 5/58 on GPU...
Processing Page 6/58 on GPU...
Processing Page 7/58 on GPU...
Processing Page 8/58 on GPU...
Processing Page 9/58 on GPU...
Processing Page 10/58 on GPU...
Processing Page 11/58 on GPU...
Processing Page 12/58 on GPU...
Processing Page 13/58 on GPU...
Processing Page 14/58 on GPU...
Processing Page 15/58 on GPU...
Processing Page 16/58 on GPU...
Processing Page 17/58 on GPU...
Processing Page 18/58 on GPU...
Processing Page 19/58 on GPU...
Processing Page 20/58 on GPU...
Processing Page 21/58 on GPU...
Processing Page 22/58 on GPU...
Processing Page 23/58 on GPU...
Processing Page 24/58 on GPU...
Processing Page 25/58 on GPU...
Processing Page 26/58 on GPU...
Processing Page 27/58 on GPU...
Processing

In [13]:
import re

def clean_arabic_ocr(text):
    # 1. Fix common high-frequency OCR misrecognitions
    replacements = {
        r'\bمسادة\b': 'مادة',
        r'\bقسانون\b': 'قانون',
        r'\bالقسانون\b': 'القانون',
        r'\bالنسواب\b': 'النواب',
        r'\bهسذا\b': 'هذا',
        r'\bعلسى\b': 'على',
        r'\bتعديسل\b': 'تعديل',
        r'\bدقم\b': 'رقم',
        r'\bدقثم\b': 'رقم',
        r'\bالمدد\b': 'العدد',
        r'\bالمفحة\b': 'الصفحة',
        r'\bالمفحق\b': 'الملحق',
        r'\bالجرانم\b': 'الجرائم',
        r'\bمجلسس\b': 'مجلس',
        r'\bعل\b(?=\s+على|\s+عن)': 'على',  # dangling truncation
    }
    
    for pattern, repl in replacements.items():
        text = re.sub(pattern, repl, text)
        
    # 2. Fix false 'س' insertion inside words (e.g., 'إعسلان' -> 'إعلان', 'المو افق' -> 'الموافق')
    # Remove unwanted space splits within words
    text = re.sub(r'(\b\w{2,})س(\w{2,}\b)', r'\1\2', text)  # removes infix 'س' if sandwiched artificially
    text = re.sub(r'(?<=\w)\s+(?=\w)', ' ', text)  # normalize whitespace
    
    # 3. Arabic Normalization (Kashida / Tatweel removal)
    text = re.sub(r'ـ+', '', text)
    
    return text

# Example usage
with open("easyocr_extracted_arabic.txt", "r", encoding="utf8") as f:
    raw_ocr = f.read()

cleaned_ocr = clean_arabic_ocr(raw_ocr)

In [14]:
with open("cleaned_ocr_output.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_ocr)

print("File saved! Refresh the 'Output' pane in the right-hand panel to download 'cleaned_ocr_output.txt'.")

File saved! Refresh the 'Output' pane in the right-hand panel to download 'cleaned_ocr_output.txt'.


In [15]:
import re

INPUT_FILE = "easyocr_extracted_arabic.txt"
OUTPUT_FILE = "easyocr_extracted_arabic_3_filtered.txt"


def filter_arabic_ocr(text: str) -> str:
  lines = text.splitlines()
  cleaned_lines = []

  for line in lines:
    stripped = line.strip()

    # 1. Retain Page Boundary Markers
    if re.match(r"^---\s*PAGE\s*\d+\s*---", stripped, re.IGNORECASE):
      cleaned_lines.append(f"\n{stripped}\n")
      continue

    # 2. Filter out repetitive running headers & footers
    # Removes standalone page numbers, "العدد (1)", and "رقم الصفحة : XX"
    if re.match(
        r"^(العدد\s*\(\s*\d+\s*\)|رقم\s*الصفحة\s*:?\s*\d+|\d+)$", stripped
    ):
      continue

    # 3. Append valid text lines
    if stripped:
      cleaned_lines.append(stripped)

  return "\n".join(cleaned_lines)


# Execute filtering pipeline
with open(INPUT_FILE, "r", encoding="utf-8") as f:
  raw_text = f.read()

filtered_content = filter_arabic_ocr(raw_text)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
  f.write(filtered_content)

print(f"Filtered output successfully saved to: {OUTPUT_FILE}")

Filtered output successfully saved to: easyocr_extracted_arabic_3_filtered.txt


In [17]:
import re
from pathlib import Path

# Paths configuration
RAW_OCR_PATH = Path("easyocr_extracted_arabic.txt")
OUTPUT_PATH = Path("cleaned_ocr_output_v3.txt")

# Read raw file
if RAW_OCR_PATH.exists():
    with open(RAW_OCR_PATH, "r", encoding="utf-8") as f:
        raw_text = f.read()
    print(f"Loaded raw OCR text successfully.")
    print(f"Total Lines: {len(raw_text.splitlines()):,}")
    print(f"Total Characters: {len(raw_text):,}\n")
    
    print("--- SAMPLE RAW LINES (First 10) ---")
    for line in raw_text.splitlines()[:10]:
        print(line)
else:
    print(f"File '{RAW_OCR_PATH}' not found. Please place the raw text file in the working directory.")

Loaded raw OCR text successfully.
Total Lines: 2,868
Total Characters: 54,771

--- SAMPLE RAW LINES (First 10) ---
--- PAGE 1 ---
مجلسس النسواب الليبي
ديوان مجلس النواب
الرسمية
العدد الأول
السنة الأولى
جمادي الثانى / 1444 هجري
/ 23
2023 /01/16
الموافق


In [19]:
def strip_headers_and_footers(text: str) -> str:
    lines = text.splitlines()
    cleaned_lines = []
    
    # Patterns matching typical Libyan Gazette headers/footers
    header_patterns = [
        r"^الجريدة\s+الرسمية.*$",
        r"^السنة\s+[\u0600-\u06FF\s]+العدد.*$",
        r"^صفحة\s+\d+.*$",
        r"^---+\s*PAGE\s*\d+\s*---+",
        r"^\d+\s*$"  # Standalone page numbers
    ]
    
    combined_pattern = re.compile("|".join(header_patterns), re.IGNORECASE)
    
    for line in lines:
        line_str = line.strip()
        # Skip matched header/footer lines
        if combined_pattern.match(line_str):
            continue
        cleaned_lines.append(line_str)
        
    # Rejoin lines while collapsing multiple empty lines
    result = "\n".join(cleaned_lines)
    result = re.sub(r"\n{3,}", "\n\n", result)
    return result

# Apply Step 1
text_step1 = strip_headers_and_footers(raw_text)

print(f"Lines after header stripping: {len(text_step1.splitlines()):,} (Reduced by {len(raw_text.splitlines()) - len(text_step1.splitlines()):,} lines)")

Lines after header stripping: 2,794 (Reduced by 74 lines)


In [20]:
def fix_deterministic_ocr_errors(text: str) -> str:
    # 1. Exact lookup replacement dictionary for common EasyOCR misreadings
    word_replacements = {
        "دقىم": "رقم",
        "الصنحة": "الصفحة",
        "الجرانم": "الجرائم",
        "النسواب": "النواب",
        "قسانون": "قانون",
        "مسادة": "مادة",
        "الاحكام": "الأحكام",
        "الانتقالية": "الانتقالية",
    }
    
    # Replace explicit word errors
    for wrong, right in word_replacements.items():
        text = re.sub(r'\b' + re.escape(wrong) + r'\b', right, text)
        
    # 2. Targeted end-of-word duplicate character fixing (e.g., مجلسس -> مجلس)
    # Target only specific doubled letters at the end of words to prevent over-cleaning
    stutter_pattern = r'(\b[\u0600-\u06FF]+)(س|ب|ت|م|ن){2}\b'
    text = re.sub(stutter_pattern, r'\1\2', text)
    
    return text

# Apply Step 3
text_step2 = fix_deterministic_ocr_errors(text_step1)
print("Step 3 complete: Deterministic OCR typo fixes applied.")

Step 3 complete: Deterministic OCR typo fixes applied.


In [21]:
def normalize_legal_formatting(text: str) -> str:
    # Fix inverted parentheses around numbers, e.g., )10( or )10000( -> (10)
    text = re.sub(r'\)(\d+)\(', r'(\1)', text)
    
    # Standardize "مادة" spacing with article numbers, e.g., مادة)1( or مادة 1 -> مادة (1)
    text = re.sub(r'مادة\s*\(?\s*(\d+)\s*\)?', r'مادة (\1)', text)
    
    # Standardize spaces around Arabic colons and punctuation
    text = re.sub(r'\s+:', ':', text)
    
    return text

# Apply Step 4
text_step3 = normalize_legal_formatting(text_step2)
print("Step 4 complete: Legal article numbers and punctuation normalized.")

Step 4 complete: Legal article numbers and punctuation normalized.


In [22]:
def validate_and_save(text: str, output_path: Path):
    # 1. Validation Check: Ensure critical words containing 'س' are intact
    critical_terms = ["السجن", "الدستوري", "الرسمية", "الاستخدام"]
    print("--- VALIDATION CHECK ---")
    for term in critical_terms:
        count = len(re.findall(r'\b' + re.escape(term) + r'\b', text))
        print(f"Term '{term}': {count} matches found.")
        
    # Check for corrupted artifacts like 'الجن' (from broken 'السجن')
    corrupted_count = len(re.findall(r'\bالجن\b', text))
    if corrupted_count > 0:
        print(f"\n⚠️ WARNING: Found {corrupted_count} instances of 'الجن'. Verify if this is corrupted 'السجن'.")
    else:
        print("\n✅ PASSED: No accidental corruption of 'السجن' into 'الجن' detected.")

    # 2. Save Cleaned Output to File
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)
        
    print(f"\nSuccessfully saved cleaned OCR output to: {output_path.resolve()}")

# Run validation and save output
validate_and_save(text_step3, OUTPUT_PATH)

--- VALIDATION CHECK ---
Term 'السجن': 7 matches found.
Term 'الدستوري': 2 matches found.
Term 'الرسمية': 12 matches found.
Term 'الاستخدام': 3 matches found.

✅ PASSED: No accidental corruption of 'السجن' into 'الجن' detected.

Successfully saved cleaned OCR output to: /kaggle/working/cleaned_ocr_output_v3.txt


In [23]:
import re
import json

def chunk_gazette_document(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Split document into individual laws
    law_patterns = re.split(r'(قانون\s+رقم\s+\(\s*\d+\s*\)\s+لسنة\s+\d{4}م?)', text)
    
    chunks = []
    
    # Process preambles and law sections
    current_law_title = "Preamble / Header"
    
    for idx, part in enumerate(law_patterns):
        if re.match(r'قانون\s+رقم\s+\(\s*\d+\s*\)\s+لسنة\s+\d{4}م?', part):
            current_law_title = re.sub(r'\s+', ' ', part.strip())
            continue
            
        # Parse Chapters and Articles within the law
        chapter_match = None
        articles = re.split(r'(مادة\s*\(\s*\d+\s*\))', part)
        
        current_article_num = None
        
        for i in range(len(articles)):
            item = articles[i].strip()
            if not item:
                continue
                
            # Check if item is an article header
            art_header = re.match(r'مادة\s*\(\s*(\d+)\s*\)', item)
            if art_header:
                current_article_num = int(art_header.group(1))
            elif current_article_num is not None:
                # Article Content
                content = item
                
                # Extract embedded article title if present on the first line
                lines = [l.strip() for l in content.split('\n') if l.strip()]
                art_title = lines[0] if lines else ""
                
                chunks.append({
                    "law_title": current_law_title,
                    "article_number": current_article_num,
                    "article_title": art_title,
                    "content": f"مادة ({current_article_num})\n" + content
                })
                current_article_num = None

    return chunks

# Run chunker
chunks = chunk_gazette_document('cleaned_ocr_output_v3.txt')

# Save to JSON
with open('chunked_gazette_output.json', 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Successfully generated {len(chunks)} article-level chunks.")

Successfully generated 127 article-level chunks.


In [25]:
!pip install chromadb sentence-transformers tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 69.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: o

In [26]:
# Install dependencies if needed:
# !pip install sentence-transformers chromadb tqdm

import json
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Load Generated Chunks
with open('chunked_gazette_output.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} article chunks.")

# 2. Initialize Arabic-Supported Multilingual Embedding Model
# BAAI/bge-m3 provides dense vector representations for Arabic legal terminology
embedding_model = SentenceTransformer("BAAI/bge-m3")

# 3. Initialize Persistent Local Vector Store
client = chromadb.PersistentClient(path="./libyan_laws_vector_db")

# Create or replace collection configured with Cosine Similarity
collection = client.get_or_create_collection(
    name="libyan_gazette_2022",
    metadata={"hnsw:space": "cosine"}
)

# 4. Prepare Batches for Ingestion
documents = [c["content"] for c in chunks]
metadatas = [
    {
        "law_title": c["law_title"],
        "article_number": int(c["article_number"]),
        "article_title": c["article_title"]
    }
    for c in chunks
]

# Generate unique deterministic identifiers per chunk
ids = [f"art_{idx+1}_law{c['article_number']}" for idx, c in enumerate(chunks)]

# 5. Generate Dense Vectors
print("Generating embeddings (BAAI/bge-m3)...")
embeddings = embedding_model.encode(
    documents, 
    show_progress_bar=True, 
    batch_size=16, 
    normalize_embeddings=True
)

# 6. Upsert Chunks + Embeddings + Metadata to ChromaDB
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadatas
)

print(f"Successfully ingested {collection.count()} chunks into the vector store.")

# 7. Verification Test Query
test_query = "ما هي عقوبة الدخول غير المشروع أو الاختراق؟"
query_embedding = embedding_model.encode([test_query], normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)

print("\n=== Verification Search Results ===")
for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    print(f"Law: {meta['law_title']}")
    print(f"Article Number: {meta['article_number']} | Title: {meta['article_title']}")
    print(f"Distance: {dist:.4f}")
    print(f"Snippet:\n{doc[:200]}...\n" + "-"*50)

Loaded 127 article chunks.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Generating embeddings (BAAI/bge-m3)...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Successfully ingested 127 chunks into the vector store.

=== Verification Search Results ===
Law: قانون رقم (1) لسنة 2005م
Article Number: 39 | Title: حيازة وسانل التشفير واستعمالها
Distance: 0.4386
Snippet:
مادة (39)
حيازة وسانل التشفير واستعمالها
يعاقب بالسجن وبغرامة لا تقل
(20000) عشرين
ألف
دينار ولا
عن
تزيد على (100000) منة
أو حاز  أو وفر  أو
دينار كل من
الف
انتج
زرع أوسوق أوصنع أوصدر  أواستورد وسانل ...
--------------------------------------------------
Law: قانون رقم (1) لسنة 2005م
Article Number: 47 | Title: التصت غير الشروع
Distance: 0.4551
Snippet:
مادة (47)
التصت غير الشروع
سنة كل من تصت
يعاقب بالحس مدة لا تقل
لصالح
شبكة المعلومات الدولية
غيره على الاتصالات
تجرى
لصالح
التي
عبر
أو اي وسيلة الكترونية

دقم الصفمة:19
العدد ) 1(
وتكون العقوبة السجن ...
--------------------------------------------------


In [27]:
def retrieve_legal_context(query: str, top_k: int = 3, law_number: str = None):
    """
    Retrieves relevant article chunks from ChromaDB using semantic similarity.
    
    Args:
        query (str): Search query in Arabic.
        top_k (int): Number of top matching articles to return.
        law_number (str): Optional law number filter (e.g., "5" or "6").
        
    Returns:
        list: Retrieved articles with metadata and distance scores.
    """
    # Vectorize query
    query_embedding = embedding_model.encode([query], normalize_embeddings=True).tolist()
    
    # Configure metadata filter if law_number is specified
    where_filter = None
    if law_number:
        # ChromaDB metadata filter
        where_filter = {"law_title": {"$contains": f"رقم ({law_number})"}}
    
    # Query collection
    query_args = {
        "query_embeddings": query_embedding,
        "n_results": top_k
    }
    if where_filter:
        query_args["where"] = where_filter

    results = collection.query(**query_args)
    
    retrieved_articles = []
    if results['documents'] and results['documents'][0]:
        for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
            retrieved_articles.append({
                "content": doc,
                "metadata": meta,
                "distance": dist
            })
            
    return retrieved_articles

# --- Test Query Execution ---
sample_query = "ما هي التزامات مقدم خدمة التوقيع الإلكتروني؟"
retrieved_docs = retrieve_legal_context(sample_query, top_k=2)

print(f"Query: {sample_query}\n" + "="*50)
for idx, doc in enumerate(retrieved_docs, 1):
    print(f"Result #{idx} | Law: {doc['metadata']['law_title']} | Article: {doc['metadata']['article_number']}")
    print(f"Cosine Distance: {doc['distance']:.4f}")
    print(f"Content Snippet:\n{doc['content'][:250]}...\n" + "-"*50)

Query: ما هي التزامات مقدم خدمة التوقيع الإلكتروني؟
Result #1 | Law: قانون رقم (1) لسنة 2005م | Article: 11
Cosine Distance: 0.3722
Content Snippet:
مادة (11)
بإنشاء توقيع الكتروني بما
يلتزم من
يلي: -
يقوم
قانوني.
استخدام أداة التوقيع استخداما
-1
غير
عدم
2- بذل العناية لتجب استخدام معلومات إنشاء توقيعه استخداما
غير
مسموح به .
أن يقوم بدون تأخير باستخدام الوسائل المتاحة له من قبل
مقدم
-3
لإخطار أي...
--------------------------------------------------
Result #2 | Law: قانون رقم (1) لسنة 2005م | Article: 8
Cosine Distance: 0.3815
Content Snippet:
مادة (8)
شخص طبيعيء أو معنويأن ينشئ توقيعًا الكترونيا عن طريق
لأي
منظومة إنشاء موثوق بهاء وتحدد اللانحة التنفيذية الشروط والمواصفات
اللازمة لذلك ....
--------------------------------------------------


In [28]:
def construct_rag_prompt(user_query: str, retrieved_chunks: list) -> str:
    """
    Formats the retrieved legal context into a strict grounded system prompt.
    """
    context_str = ""
    for idx, chunk in enumerate(retrieved_chunks, 1):
        context_str += f"\n--- [المستند {idx}] ---\n"
        context_str += f"القانون: {chunk['metadata']['law_title']}\n"
        context_str += f"{chunk['content']}\n"
    
    system_instruction = (
        "أنت مساعد قانوني متخصص في التشريعات الليبية. إجابتك يجب أن تستند تماماً "
        "وعلى نحو حصري إلى المواد القانونية المرفقة أدناه. لا تقم بافتراض أي أحكام غير مذكورة. "
        "عند الإجابة، اذكر اسم القانون ورقم المادة النصية المتعلقة بالسؤال."
    )
    
    formatted_prompt = f"""{system_instruction}

المواد القانونية المتاحة:
{context_str}

سؤال المستخدم: {user_query}

الإجابة القانونية المستندة للمواد المرفقة:"""
    
    return formatted_prompt

# --- Test Prompt Construction ---
rag_prompt = construct_rag_prompt(sample_query, retrieved_docs)
print("=== Constructed Prompt Preview ===")
print(rag_prompt[:800] + "\n\n[... Context Truncated for Preview ...]")

=== Constructed Prompt Preview ===
أنت مساعد قانوني متخصص في التشريعات الليبية. إجابتك يجب أن تستند تماماً وعلى نحو حصري إلى المواد القانونية المرفقة أدناه. لا تقم بافتراض أي أحكام غير مذكورة. عند الإجابة، اذكر اسم القانون ورقم المادة النصية المتعلقة بالسؤال.

المواد القانونية المتاحة:

--- [المستند 1] ---
القانون: قانون رقم (1) لسنة 2005م
مادة (11)
بإنشاء توقيع الكتروني بما
يلتزم من
يلي: -
يقوم
قانوني.
استخدام أداة التوقيع استخداما
-1
غير
عدم
2- بذل العناية لتجب استخدام معلومات إنشاء توقيعه استخداما
غير
مسموح به .
أن يقوم بدون تأخير باستخدام الوسائل المتاحة له من قبل
مقدم
-3
لإخطار أيشخص يتوقع أنه
جهودًا
خدمات التصديق  وأن يبذل
سيعتمد أويقدم خدمات استنادا إلى توقيعه الإلكترونيء إذاكان
الموقع
يعلم ان أداة إنشاء التوقيع قدم الإخلال بهاء أوكات الظروف

دفم الصفعة:29
العدد ) ا (
بوجود شبهات كبيرة في احتمال تعرض
المعلومة لدى الم

[... Context Truncated for Preview ...]


In [29]:
import re

def sanitize_arabic_ocr(text: str) -> str:
    # 1. Remove page numbers and gazette headers/footers
    text = re.sub(r'دفم الصفعة\s*:\s*\d+', '', text)
    text = re.sub(r'رقم الصفحة\s*:\s*\d+', '', text)
    text = re.sub(r'العدد\s*\(\s*[\dأa-zA-Z]+\s*\)', '', text)
    text = re.sub(r'السنة\s+[\dأa-zA-Z]+', '', text)
    text = re.sub(r'الجريدة الرسمية', '', text)

    # 2. Fix broken multi-line sentences (rejoin lines that don't end with sentence terminators)
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        line_str = line.strip()
        if not line_str:
            continue
            
        # Keep section headers and article definitions separate
        if re.match(r'^(مادة|قانون|الفصل|الباب|الديباجة)', line_str):
            cleaned_lines.append(f"\n{line_str}\n")
        else:
            cleaned_lines.append(line_str)

    # Rejoin wrapped lines into full sentences
    rejoined_text = " ".join(cleaned_lines)
    
    # Normalize excessive whitespaces
    rejoined_text = re.sub(r'\s+', ' ', rejoined_text)
    
    # Format article headers cleanly on new lines
    rejoined_text = re.sub(r'\s*(مادة\s*\(\s*\d+\s*\))\s*', r'\n\n\1\n', rejoined_text)
    rejoined_text = re.sub(r'\s*(قانون\s+رقم\s+\(\s*\d+\s*\)\s+لسنة\s+\d{4}م?)\s*', r'\n\n\1\n', rejoined_text)

    return rejoined_text

# Clean your raw text file before chunking
with open('cleaned_ocr_output_v3.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

clean_text = sanitize_arabic_ocr(raw_text)

with open('sanitized_gazette_text.txt', 'w', encoding='utf-8') as f:
    f.write(clean_text)

print("Text sanitization complete. Re-run chunking on 'sanitized_gazette_text.txt'.")

Text sanitization complete. Re-run chunking on 'sanitized_gazette_text.txt'.


In [30]:
pip install faiss-cpu sentence-transformers langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [34]:
!pip install -q faiss-gpu sentence-transformers langchain-community langchain-core

In [35]:
import re
import torch
import faiss
from typing import List
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 1. Regex Chunking Function
def parse_libyan_laws(raw_text: str) -> List[Document]:
    documents = []
    law_pattern = r"(قانون\s+رقم\s*\(?\d+\)?\s+لسنة\s+\d{4}\b.*?)(?=(?:قانون\s+رقم\s*\(?\d+\)?\s+لسنة\s+\d{4}\b)|$)"
    laws = re.findall(law_pattern, raw_text, flags=re.DOTALL | re.UNICODE) or [raw_text]

    for law_text in laws:
        law_num_match = re.search(r"قانون\s+رقم\s*\(?(\d+)\)?\s+لسنة\s+(\d{4})", law_text)
        law_number = law_num_match.group(1) if law_num_match else "غير محدد"
        law_year = law_num_match.group(2) if law_num_match else "2022"
        
        title_match = re.search(r"بشأن\s+([^\n.]+)", law_text)
        law_title = title_match.group(0).strip() if title_match else "تشريع ليبي"

        article_pattern = r"(مادة\s*\(?\d+\)?\b[\s\S]*?)(?=(?:مادة\s*\(?\d+\)?\b)|$)"
        articles = re.findall(article_pattern, law_text, flags=re.MULTILINE)

        for art_text in articles:
            art_text_cleaned = art_text.strip()
            if not art_text_cleaned:
                continue

            art_num_match = re.search(r"مادة\s*\(?(\d+)\)?", art_text_cleaned)
            article_number = int(art_num_match.group(1)) if art_num_match else None

            documents.append(
                Document(
                    page_content=art_text_cleaned,
                    metadata={
                        "law_number": law_number,
                        "law_year": law_year,
                        "law_title": law_title,
                        "article_number": article_number,
                        "source": f"قانون {law_number} لسنة {law_year}",
                        "char_count": len(art_text_cleaned)
                    }
                )
            )

    return documents


# 2. Load Embeddings Model directly on GPU
model_name = "BAAI/bge-m3"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': device},
    encode_kwargs={
        'normalize_embeddings': True,
        'batch_size': 64  # Increased batch size for GPU parallelization
    }
)


# 3. Build FAISS Vector Store on GPU
def create_faiss_vector_store_gpu(documents: List[Document], save_path: str = "faiss_libyan_laws"):
    print(f"Building embeddings on {device.upper()} for {len(documents)} articles...")
    
    # Generate index via GPU embeddings
    vectorstore = FAISS.from_documents(documents, embeddings)

    # Move FAISS index memory directly to GPU CUDA resources
    if torch.cuda.is_available():
        res = faiss.StandardGpuResources()
        gpu_index = faiss.index_cpu_to_gpu(res, 0, vectorstore.index)
        vectorstore.index = gpu_index
        print("FAISS index successfully moved to GPU.")

    # Save to disk (FAISS converts GPU index back to CPU automatically for saving)
    vectorstore.save_local(save_path)
    print(f"Index persisted locally to `{save_path}`.")
    return vectorstore


# Execution
if __name__ == "__main__":
    sample_text = """
    قانون رقم (5) لسنة 2022 م بشأن مكافحة الجرائم الإلكترونية
    مادة (1) تعاريف: في تطبيق أحكام هذا القانون يقصد بالكلمات والعبارات التالية...
    مادة (12) يعاقب بالحبس مدة لا تقل عن سنة وبغرامة لا تقل عن ألف دينار كل من دخل عمداً وبدون وجه حق إلى نظام أو موقع إلكتروني.
    """
    
    docs = parse_libyan_laws(sample_text)
    vector_db = create_faiss_vector_store_gpu(docs)

/tmp/ipykernel_46/171009895.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


Using device: cuda (Tesla T4)


/tmp/ipykernel_46/171009895.py:57: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Building embeddings on CUDA for 2 articles...
FAISS index successfully moved to GPU.


RuntimeError: Error in void faiss::write_index(const Index*, IOWriter*, int) at /project/faiss/impl/index_write.cpp:1226: don't know how to serialize this type of index

In [36]:
!pip install -U -q faiss-gpu langchain-huggingface langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 14.3 MB/s eta 0:00:0000:01


In [37]:
import re
import torch
import faiss
from typing import List
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Confirm GPU Availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


# 1. Structural Regex Chunking
def parse_libyan_laws(raw_text: str) -> List[Document]:
    documents = []
    law_pattern = r"(قانون\s+رقم\s*\(?\d+\)?\s+لسنة\s+\d{4}\b.*?)(?=(?:قانون\s+رقم\s*\(?\d+\)?\s+لسنة\s+\d{4}\b)|$)"
    laws = re.findall(law_pattern, raw_text, flags=re.DOTALL | re.UNICODE) or [raw_text]

    for law_text in laws:
        law_num_match = re.search(r"قانون\s+رقم\s*\(?(\d+)\)?\s+لسنة\s+(\d{4})", law_text)
        law_number = law_num_match.group(1) if law_num_match else "غير محدد"
        law_year = law_num_match.group(2) if law_num_match else "2022"
        
        title_match = re.search(r"بشأن\s+([^\n.]+)", law_text)
        law_title = title_match.group(0).strip() if title_match else "تشريع ليبي"

        article_pattern = r"(مادة\s*\(?\d+\)?\b[\s\S]*?)(?=(?:مادة\s*\(?\d+\)?\b)|$)"
        articles = re.findall(article_pattern, law_text, flags=re.MULTILINE)

        for art_text in articles:
            art_text_cleaned = art_text.strip()
            if not art_text_cleaned:
                continue

            art_num_match = re.search(r"مادة\s*\(?(\d+)\)?", art_text_cleaned)
            article_number = int(art_num_match.group(1)) if art_num_match else None

            documents.append(
                Document(
                    page_content=art_text_cleaned,
                    metadata={
                        "law_number": law_number,
                        "law_year": law_year,
                        "law_title": law_title,
                        "article_number": article_number,
                        "source": f"قانون {law_number} لسنة {law_year}",
                        "char_count": len(art_text_cleaned)
                    }
                )
            )

    return documents


# 2. Updated Hugging Face Embeddings on CUDA
model_name = "BAAI/bge-m3"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': device},
    encode_kwargs={
        'normalize_embeddings': True,
        'batch_size': 64
    }
)


# 3. Vector Store Creation & Disk Persistence
def create_faiss_vector_store_gpu(documents: List[Document], save_path: str = "faiss_libyan_laws"):
    print(f"Generating GPU embeddings for {len(documents)} items...")
    
    # 1. Embeddings generated via GPU, vectors populated into CPU FAISS index
    vectorstore = FAISS.from_documents(documents, embeddings)

    # 2. Persist CPU FAISS Index to Disk (Prevents Serialization RuntimeError)
    vectorstore.save_local(save_path)
    print(f"Index successfully saved to disk at `{save_path}`.")

    # 3. Move Index to GPU Memory for In-RAM Queries
    if torch.cuda.is_available():
        res = faiss.StandardGpuResources()
        vectorstore.index = faiss.index_cpu_to_gpu(res, 0, vectorstore.index)
        print("FAISS index loaded into GPU VRAM for active querying.")

    return vectorstore


# Execution
if __name__ == "__main__":
    sample_text = """
    قانون رقم (5) لسنة 2022 م بشأن مكافحة الجرائم الإلكترونية
    مادة (1) تعاريف: في تطبيق أحكام هذا القانون يقصد بالكلمات والعبارات التالية...
    مادة (12) يعاقب بالحبس مدة لا تقل عن سنة وبغرامة لا تقل عن ألف دينار كل من دخل عمداً وبدون وجه حق إلى نظام أو موقع إلكتروني.
    """
    
    docs = parse_libyan_laws(sample_text)
    vector_db = create_faiss_vector_store_gpu(docs)

Using device: cuda (Tesla T4)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Generating GPU embeddings for 2 items...
Index successfully saved to disk at `faiss_libyan_laws`.
FAISS index loaded into GPU VRAM for active querying.


In [38]:
!pip install -q sentence-transformers transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.9 MB/s eta 0:00:00:00:0100:01


In [39]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import FAISS

# 1. Load the Cross-Encoder Re-ranker
print("Loading BGE-M3 Re-ranker...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda" if torch.cuda.is_available() else "cpu")

# 2. Load LLM (Qwen2.5-7B-Instruct 4-bit to fit Tesla T4 VRAM alongside embeddings)
model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading LLM: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. Retrieval & Re-ranking Engine
def retrieve_and_rerank(query: str, vector_db: FAISS, top_k_retrieval: int = 10, top_n_rerank: int = 3):
    # Step A: Dense Vector Search
    initial_docs = vector_db.similarity_search(query, k=top_k_retrieval)
    
    if not initial_docs:
        return []

    # Step B: Prepare pairs for Cross-Encoder scoring
    pairs = [[query, doc.page_content] for doc in initial_docs]
    scores = reranker.predict(pairs)

    # Step C: Sort docs by re-ranker score
    scored_docs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)
    
    # Return top N re-ranked documents
    reranked_docs = [doc for doc, score in scored_docs[:top_n_rerank]]
    return reranked_docs


# 4. RAG Generation Pipeline
def answer_legal_query(query: str, vector_db: FAISS):
    # Retrieve top relevant context
    context_docs = retrieve_and_rerank(query, vector_db)
    
    # Format context with law numbers and article metadata
    context_str = ""
    for idx, doc in enumerate(context_docs, 1):
        context_str += f"\n--- المصدر [{idx}]: {doc.metadata.get('source', '')} - مادة ({doc.metadata.get('article_number', '')}) ---\n"
        context_str += f"{doc.page_content}\n"

    # Construct System & User Prompts
    system_prompt = (
        "أنت مستشار قانوني متخصص في التشريعات والقوانين الليبية.\n"
        "أجب عن سؤال المستخدم بوضوح ودقة متناهية بالاعتماد **فقط** على النصوص القانونية المرفقة في السياق.\n"
        "القواعد المتبعة:\n"
        "1. اذكر رقم القانون ورقم المادة بدقة عند استشهادك بالعقوبة أو الحكم.\n"
        "2. إذا لم تجد الإجابة صراحة في النصوص المرفقة، قل: 'لا تتوفر معلومات كافية في النصوص المتاحة.' دون اختلاق أحكام."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"السياق القانوني المتاح:\n{context_str}\n\nسؤال المستخدم: {query}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,  # Low temperature for factual precision
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response, context_docs


# 5. Execution Test
if __name__ == "__main__":
    user_query = "ما هي عقوبة الدخول غير المشروع إلى موقع إلكتروني؟"
    
    answer, retrieved_sources = answer_legal_query(user_query, vector_db)
    
    print("\n================== الإجابة المولدة ==================")
    print(answer)
    print("\n================== المصادر المسترجعة ==================")
    for doc in retrieved_sources:
        print(f"• {doc.metadata['source']} | مادة ({doc.metadata['article_number']})")

Loading BGE-M3 Re-ranker...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


================== الإجابة المولدة ==================
فقعوبة الدخول غير المشروع إلى نظام أو موقع إلكتروني وفقاً للقانون 5 لسنة 2022 - مادة (12) هي الحبس مدة لا تقل عن سنة وبغرامة لا تقل عن ألف دينار.

================== المصادر المسترجعة ==================
• قانون 5 لسنة 2022 | مادة (12)
• قانون 5 لسنة 2022 | مادة (1)


In [40]:
!pip install -q rank-bm25

In [41]:
import numpy as np
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any

# 1. Define Benchmark Dataset (Queries + Ground Truth Article IDs)
eval_dataset = [
    {
        "query": "ما هي عقوبة اختراق نظام معلوماتي؟",
        "relevant_articles": [12]  # Article 12 of Law 5
    },
    {
        "query": "هل التوقيع الالكتروني له حجة قانونية في المعاملات التجارة؟",
        "relevant_articles": [5]   # Article 5 of Law 6
    }
]

# 2. Tokenizer for BM25 Keyword Search
def arabic_tokenize(text: str) -> List[str]:
    # Simple word tokenization for BM25
    return text.lower().split()

# Prepare corpus for BM25
corpus = [doc.page_content for doc in docs]
tokenized_corpus = [arabic_tokenize(doc) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


# 3. Search Functions
def search_bm25(query: str, top_k: int = 2) -> List[int]:
    tokenized_query = arabic_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [docs[i].metadata["article_number"] for i in top_indices if docs[i].metadata["article_number"] is not None]

def search_semantic_faiss(query: str, vector_db, top_k: int = 2) -> List[int]:
    results = vector_db.similarity_search(query, k=top_k)
    return [doc.metadata["article_number"] for doc in results if doc.metadata.get("article_number") is not None]


# 4. Metric Computation Logic
def compute_metrics(retrieved_ids: List[int], ground_truth_ids: List[int], k: int):
    retrieved_at_k = retrieved_ids[:k]
    hits = set(retrieved_at_k).intersection(set(ground_truth_ids))
    
    precision = len(hits) / k if k > 0 else 0.0
    recall = len(hits) / len(ground_truth_ids) if len(ground_truth_ids) > 0 else 0.0
    
    # Reciprocal Rank (RR)
    mrr = 0.0
    for idx, doc_id in enumerate(retrieved_at_k, 1):
        if doc_id in ground_truth_ids:
            mrr = 1.0 / idx
            break
            
    return precision, recall, mrr


# 5. Benchmark Execution
def run_evaluation(vector_db, k: int = 2):
    bm25_p, bm25_r, bm25_mrr = [], [], []
    faiss_p, faiss_r, faiss_mrr = [], [], []

    for item in eval_dataset:
        q = item["query"]
        gt = item["relevant_articles"]

        # Run BM25
        bm25_res = search_bm25(q, top_k=k)
        p, r, mrr = compute_metrics(bm25_res, gt, k)
        bm25_p.append(p); bm25_r.append(r); bm25_mrr.append(mrr)

        # Run FAISS Semantic Search
        faiss_res = search_semantic_faiss(q, vector_db, top_k=k)
        p, r, mrr = compute_metrics(faiss_res, gt, k)
        faiss_p.append(p); faiss_r.append(r); faiss_mrr.append(mrr)

    print("================== Evaluation Results ==================")
    print(f"Metrics @ K={k}\n")
    print(f"BM25 Keyword Search  -> Precision: {np.mean(bm25_p):.2f} | Recall: {np.mean(bm25_r):.2f} | MRR: {np.mean(bm25_mrr):.2f}")
    print(f"FAISS Semantic Search -> Precision: {np.mean(faiss_p):.2f} | Recall: {np.mean(faiss_r):.2f} | MRR: {np.mean(faiss_mrr):.2f}")

# Execute
run_evaluation(vector_db, k=2)

================== Evaluation Results ==================
Metrics @ K=2

BM25 Keyword Search  -> Precision: 0.25 | Recall: 0.50 | MRR: 0.50
FAISS Semantic Search -> Precision: 0.25 | Recall: 0.50 | MRR: 0.50


In [42]:
# Install required rank-bm25 library if not present
!pip install -q rank-bm25 pandas tabulate

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from typing import List, Dict, Tuple, Any

# Ensure docs and vector_db exist from previous steps
assert 'docs' in globals() and len(docs) > 0, "Error: 'docs' list is empty. Run chunking first."
assert 'vector_db' in globals(), "Error: 'vector_db' not found. Run vector store creation first."
assert 'reranker' in globals(), "Error: 'reranker' not found. Load BGE CrossEncoder first."

print(f"Loaded {len(docs)} legal chunks from the 58-page Official Gazette for evaluation.\n")

# ==========================================
# 1. Unique Document Identifier Helper
# ==========================================
def get_doc_id(doc) -> str:
    """Creates a unique string key (e.g., 'Law5_Art12') to uniquely identify articles."""
    law = doc.metadata.get("law_number", "Unknown")
    art = doc.metadata.get("article_number", "Unknown")
    return f"Law{law}_Art{art}"

# Build corpus lookup map
doc_id_corpus = [get_doc_id(d) for d in docs]


# ==========================================
# 2. Setup BM25 Keyword Index
# ==========================================
def arabic_tokenizer(text: str) -> List[str]:
    # Simple whitespace + basic cleaning tokenization
    return text.strip().lower().split()

tokenized_corpus = [arabic_tokenizer(d.page_content) for d in docs]
bm25_index = BM25Okapi(tokenized_corpus)


# ==========================================
# 3. Retrieval Strategy Wrappers
# ==========================================
def bm25_retrieve(query: str, top_k: int) -> List[str]:
    tokenized_q = arabic_tokenizer(query)
    scores = bm25_index.get_scores(tokenized_q)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [doc_id_corpus[i] for i in top_indices]

def faiss_retrieve(query: str, top_k: int) -> List[str]:
    retrieved_docs = vector_db.similarity_search(query, k=top_k)
    return [get_doc_id(d) for d in retrieved_docs]

def faiss_rerank_retrieve(query: str, candidate_k: int = 15, final_k: int = 3) -> List[str]:
    initial_docs = vector_db.similarity_search(query, k=candidate_k)
    pairs = [[query, d.page_content] for d in initial_docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)
    return [get_doc_id(d) for d, _ in scored_docs[:final_k]]


# ==========================================
# 4. Evaluation Metrics Computation
# ==========================================
def calculate_metrics(retrieved_ids: List[str], ground_truth_ids: List[str], k: int) -> Tuple[float, float, float]:
    retrieved_at_k = retrieved_ids[:k]
    hits = set(retrieved_at_k).intersection(set(ground_truth_ids))
    
    # Precision@K: Ratio of relevant retrieved docs over K
    precision = len(hits) / k if k > 0 else 0.0
    
    # Recall@K: Ratio of relevant retrieved docs over total ground truth
    recall = len(hits) / len(ground_truth_ids) if len(ground_truth_ids) > 0 else 0.0
    
    # MRR (Mean Reciprocal Rank): Rank position of the first relevant document
    mrr = 0.0
    for idx, doc_id in enumerate(retrieved_at_k, start=1):
        if doc_id in ground_truth_ids:
            mrr = 1.0 / idx
            break
            
    return precision, recall, mrr


# ==========================================
# 5. Benchmark Query Dataset
# ==========================================
# Customize 'ground_truth' IDs according to your specific law numbers & article numbers parsed from the gazette
eval_benchmark = [
    {
        "query_type": "Exact Match (Keyword)",
        "query": "ما هي تعاريف وأحكام الجرائم الإلكترونية حسب المادة 1؟",
        "ground_truth": ["Law5_Art1"]
    },
    {
        "query_type": "Semantic Mismatch (Synonyms / Colloquial)",
        "query": "ما هي عقوبة الهاكرز والقرصنة واختراق موقع إلكتروني؟",
        "ground_truth": ["Law5_Art12"]
    },
    {
        "query_type": "Legal Concept Query",
        "query": "هل التوقيع أو الإمضاء الرقمي يتساوى مع الخط اليدوي في المعاملات؟",
        "ground_truth": ["Law6_Art5"]
    },
    {
        "query_type": "Penalty / Fine Query",
        "query": "ما هي عقوبة الاستيلاء غير المشروع على أموال أو توقيع إلكتروني؟",
        "ground_truth": ["Law5_Art4"]
    }
]


# ==========================================
# 6. Run Evaluation & Generate Table
# ==========================================
def run_full_evaluation(k_values: List[int] = [3, 5]):
    results_summary = []

    for k in k_values:
        bm25_p, bm25_r, bm25_mrr = [], [], []
        faiss_p, faiss_r, faiss_mrr = [], [], []
        rerank_p, rerank_r, rerank_mrr = [], [], []

        for item in eval_benchmark:
            query = item["query"]
            gt = item["ground_truth"]

            # 1. BM25
            bm25_ids = bm25_retrieve(query, top_k=k)
            p, r, mrr = calculate_metrics(bm25_ids, gt, k)
            bm25_p.append(p); bm25_r.append(r); bm25_mrr.append(mrr)

            # 2. FAISS Dense
            faiss_ids = faiss_retrieve(query, top_k=k)
            p, r, mrr = calculate_metrics(faiss_ids, gt, k)
            faiss_p.append(p); faiss_r.append(r); faiss_mrr.append(mrr)

            # 3. FAISS + Reranker
            rerank_ids = faiss_rerank_retrieve(query, candidate_k=15, final_k=k)
            p, r, mrr = calculate_metrics(rerank_ids, gt, k)
            rerank_p.append(p); rerank_r.append(r); rerank_mrr.append(mrr)

        # Store mean metrics
        results_summary.append({
            "Evaluation @ K": f"K = {k}",
            "Strategy": "BM25 (Keyword)",
            "Precision@K": f"{np.mean(bm25_p):.3f}",
            "Recall@K": f"{np.mean(bm25_r):.3f}",
            "MRR": f"{np.mean(bm25_mrr):.3f}"
        })
        results_summary.append({
            "Evaluation @ K": f"K = {k}",
            "Strategy": "FAISS (Dense Semantic)",
            "Precision@K": f"{np.mean(faiss_p):.3f}",
            "Recall@K": f"{np.mean(faiss_r):.3f}",
            "MRR": f"{np.mean(faiss_mrr):.3f}"
        })
        results_summary.append({
            "Evaluation @ K": f"K = {k}",
            "Strategy": "FAISS + BGE Reranker",
            "Precision@K": f"{np.mean(rerank_p):.3f}",
            "Recall@K": f"{np.mean(rerank_r):.3f}",
            "MRR": f"{np.mean(rerank_mrr):.3f}"
        })

    df_results = pd.DataFrame(results_summary)
    return df_results

# Execute Evaluation
eval_df = run_full_evaluation(k_values=[3, 5])

print("=======================================================================")
print("             RAG INFORMATION RETRIEVAL EVALUATION REPORT               ")
print("=======================================================================")
print(eval_df.to_string(index=False))

Loaded 2 legal chunks from the 58-page Official Gazette for evaluation.

             RAG INFORMATION RETRIEVAL EVALUATION REPORT               
Evaluation @ K               Strategy Precision@K Recall@K   MRR
         K = 3         BM25 (Keyword)       0.167    0.500 0.375
         K = 3 FAISS (Dense Semantic)       0.167    0.500 0.375
         K = 3   FAISS + BGE Reranker       0.167    0.500 0.500
         K = 5         BM25 (Keyword)       0.100    0.500 0.375
         K = 5 FAISS (Dense Semantic)       0.100    0.500 0.375
         K = 5   FAISS + BGE Reranker       0.100    0.500 0.500


In [43]:
# 1. Read full 58-page OCR text from disk
ocr_file_path = "/kaggle/working/sanitized_gazette_text.txt"  # Adjust filename to match your directory

with open(ocr_file_path, "r", encoding="utf-8") as f:
    full_gazette_text = f.read()

print(f"Total characters loaded from OCR text: {len(full_gazette_text):,}")

# 2. Re-run regex chunker on the FULL text
docs = parse_libyan_laws(full_gazette_text)
print(f"Successfully generated {len(docs)} legal article chunks across 58 pages!")

Total characters loaded from OCR text: 53,136
Successfully generated 127 legal article chunks across 58 pages!


In [44]:
# Re-build FAISS GPU Index with all chunks
vector_db = create_faiss_vector_store_gpu(docs, save_path="faiss_libyan_laws_58p")

Generating GPU embeddings for 127 items...
Index successfully saved to disk at `faiss_libyan_laws_58p`.
FAISS index loaded into GPU VRAM for active querying.


In [46]:
# Verify chunk count
print(f"Confirming corpus size: {len(docs)} documents in memory.")

# Re-run the evaluation script
eval_df = run_full_evaluation(k_values=[3, 5])

Confirming corpus size: 127 documents in memory.


In [47]:
# Check how IDs are formatted across your 127 parsed chunks
unique_ids = [get_doc_id(doc) for doc in docs]

print(f"Total Chunks: {len(unique_ids)}")
print("\nSample Generated IDs from your corpus:")
print(unique_ids[:15])  # Print first 15 IDs

# Check if any IDs match your evaluation benchmark targets
targets = ["Law5_Art1", "Law5_Art12", "Law6_Art5", "Law5_Art4"]
print("\nTarget Matching Check:")
for target in targets:
    matches = [i for i, doc_id in enumerate(unique_ids) if doc_id == target]
    print(f"Target '{target}' found in corpus: {len(matches)} time(s) (Indices: {matches})")

Total Chunks: 127

Sample Generated IDs from your corpus:
['Law22_Art1', 'Law22_Art2', 'Law22_Art3', 'Law22_Art4', 'Law22_Art5', 'Law22_Art7', 'Law22_Art10', 'Law22_Art11', 'Law22_Art12', 'Law22_Art13', 'Law22_Art14', 'Law22_Art15', 'Law22_Art16', 'Law22_Art17', 'Law22_Art21']

Target Matching Check:
Target 'Law5_Art1' found in corpus: 0 time(s) (Indices: [])
Target 'Law5_Art12' found in corpus: 0 time(s) (Indices: [])
Target 'Law6_Art5' found in corpus: 0 time(s) (Indices: [])
Target 'Law5_Art4' found in corpus: 0 time(s) (Indices: [])


In [49]:
import re
from typing import List
from langchain_core.documents import Document

# ==========================================
# 1. Digit Normalization Helper Function
# ==========================================
def normalize_digits(text: str) -> str:
    """Converts Eastern Arabic-Indic digits (٠١٢٣٤٥٦٧٨٩) to standard Western digits (0123456789)."""
    if not text:
        return ""
    arabic_indic = "٠١٢٣٤٥٦٧٨٩"
    western = "0123456789"
    trans = str.maketrans(arabic_indic, western)
    return str(text).translate(trans)


# ==========================================
# 2. Fixed Libyan Laws Parser
# ==========================================
def parse_libyan_laws_fixed(raw_text: str) -> List[Document]:
    documents = []
    
    # Split text into laws using law header boundaries
    law_pattern = r"(قانون\s+رقم\s*\(?\d+\)?\s+لسنة\s+\d{4}\b.*?)(?=(?:قانون\s+رقم\s*\(?\d+\)?\s+لسنة\s+\d{4}\b)|$)"
    laws = re.findall(law_pattern, raw_text, flags=re.DOTALL | re.UNICODE) or [raw_text]

    for law_text in laws:
        # Match law number (e.g., 5 or 6) separately from year (e.g., 2022)
        law_num_match = re.search(r"قانون\s+رقم\s*\(?\s*(\d+)\s*\)?\s+لسنة\s+(\d{4})", law_text)
        
        if law_num_match:
            law_number = normalize_digits(law_num_match.group(1))
            law_year = normalize_digits(law_num_match.group(2))
        else:
            law_number = "Unknown"
            law_year = "2022"
        
        # Match articles within the current law block
        article_pattern = r"(مادة\s*\(?\d+\)?\b[\s\S]*?)(?=(?:مادة\s*\(?\d+\)?\b)|$)"
        articles = re.findall(article_pattern, law_text, flags=re.MULTILINE)

        for art_text in articles:
            art_text_cleaned = art_text.strip()
            if not art_text_cleaned:
                continue

            art_num_match = re.search(r"مادة\s*\(?\s*(\d+)\s*\)?", art_text_cleaned)
            article_number = normalize_digits(art_num_match.group(1)) if art_num_match else "Unknown"

            documents.append(
                Document(
                    page_content=art_text_cleaned,
                    metadata={
                        "law_number": law_number,
                        "law_year": law_year,
                        "article_number": article_number,
                        "source": f"قانون {law_number} لسنة {law_year}",
                        "char_count": len(art_text_cleaned)
                    }
                )
            )

    return documents


# ==========================================
# 3. Robust ID Key Generator
# ==========================================
def get_robust_doc_id(doc) -> str:
    law_num = normalize_digits(str(doc.metadata.get("law_number", "")))
    art_num = normalize_digits(str(doc.metadata.get("article_number", "")))
    
    law_clean = re.sub(r"\D", "", law_num) or "Unknown"
    art_clean = re.sub(r"\D", "", art_num) or "Unknown"
    
    return f"Law{law_clean}_Art{art_clean}"


# ==========================================
# 4. Re-parsing Execution
# ==========================================
docs = parse_libyan_laws_fixed(full_gazette_text)
doc_id_corpus = [get_robust_doc_id(d) for d in docs]

print(f"Successfully re-parsed {len(docs)} legal article chunks!")
print("Sample Fixed IDs:", doc_id_corpus[:15])

Successfully re-parsed 127 legal article chunks!
Sample Fixed IDs: ['Law22_Art1', 'Law22_Art2', 'Law22_Art3', 'Law22_Art4', 'Law22_Art5', 'Law22_Art7', 'Law22_Art10', 'Law22_Art11', 'Law22_Art12', 'Law22_Art13', 'Law22_Art14', 'Law22_Art15', 'Law22_Art16', 'Law22_Art17', 'Law22_Art21']


In [50]:
import re
from typing import List
from langchain_core.documents import Document

def parse_libyan_laws_v3(raw_text: str) -> List[Document]:
    documents = []
    
    # Split by law headers
    law_pattern = r"(قانون\s+رقم\s*\(?\d+\)?\s+لسنة\s+(?:\d{4}|\d{2})\b.*?)(?=(?:قانون\s+رقم\s*\(?\d+\)?\s+لسنة\s+(?:\d{4}|\d{2})\b)|$)"
    laws = re.findall(law_pattern, raw_text, flags=re.DOTALL | re.UNICODE) or [raw_text]

    for law_text in laws:
        # Match Law Number explicitly (group 1) and Year (group 2)
        law_num_match = re.search(r"قانون\s+رقم\s*\(?\s*(\d+)\s*\)?\s+لسنة\s+(\d{2,4})", law_text)
        
        if law_num_match:
            raw_law_num = law_num_match.group(1)
            raw_year = law_num_match.group(2)
            
            # If group 1 accidentally captured the year 22 or 2022, swap or fix
            law_number = normalize_digits(raw_law_num)
            law_year = normalize_digits(raw_year)
        else:
            law_number = "Unknown"
            law_year = "2022"
        
        # Split articles within the law
        article_pattern = r"(مادة\s*\(?\d+\)?\b[\s\S]*?)(?=(?:مادة\s*\(?\d+\)?\b)|$)"
        articles = re.findall(article_pattern, law_text, flags=re.MULTILINE)

        for art_text in articles:
            art_text_cleaned = art_text.strip()
            if not art_text_cleaned:
                continue

            art_num_match = re.search(r"مادة\s*\(?\s*(\d+)\s*\)?", art_text_cleaned)
            article_number = normalize_digits(art_num_match.group(1)) if art_num_match else "Unknown"

            documents.append(
                Document(
                    page_content=art_text_cleaned,
                    metadata={
                        "law_number": law_number,
                        "law_year": law_year,
                        "article_number": article_number,
                        "source": f"قانون {law_number} لسنة {law_year}",
                        "char_count": len(art_text_cleaned)
                    }
                )
            )

    return documents

# Re-parse docs
docs = parse_libyan_laws_v3(full_gazette_text)
doc_id_corpus = [get_robust_doc_id(d) for d in docs]

print(f"Total Chunks Parsed: {len(docs)}")
print("Sample Fixed IDs:", doc_id_corpus[:15])

# Count distinct laws found
distinct_laws = set(d.metadata.get("law_number") for d in docs)
print(f"Distinct Law Numbers Detected in Corpus: {distinct_laws}")

Total Chunks Parsed: 127
Sample Fixed IDs: ['Law22_Art1', 'Law22_Art2', 'Law22_Art3', 'Law22_Art4', 'Law22_Art5', 'Law22_Art7', 'Law22_Art10', 'Law22_Art11', 'Law22_Art12', 'Law22_Art13', 'Law22_Art14', 'Law22_Art15', 'Law22_Art16', 'Law22_Art17', 'Law22_Art21']
Distinct Law Numbers Detected in Corpus: {'22'}


In [52]:
import re
from typing import List
from langchain_core.documents import Document

def parse_libyan_laws_v4(raw_text: str) -> List[Document]:
    documents = []
    
    # 1. Regex to split full text into distinct Law blocks based on their title header
    # Matches "قانون رقم (X) لسنة Y" specifically at block beginnings
    law_title_pattern = r"(قانون\s+رقم\s*\(?\s*\d+\s*\)?\s+لسنة\s+\d{2,4}[^\n]*)"
    
    # Split text by Law Headers
    splits = re.split(law_title_pattern, raw_text)
    
    # Combine split headers with their respective body chunks
    law_blocks = []
    if len(splits) > 1:
        for i in range(1, len(splits), 2):
            header = splits[i]
            body = splits[i+1] if (i+1) < len(splits) else ""
            law_blocks.append((header, header + body))
    else:
        law_blocks = [("", raw_text)]

    for header, law_text in law_blocks:
        # Extract law number & year strictly from the LAW HEADER line (not referenced background laws)
        law_num_match = re.search(r"قانون\s+رقم\s*\(?\s*(\d+)\s*\)?\s+لسنة\s+(\d{2,4})", header)
        
        if law_num_match:
            law_number = law_num_match.group(1)
            law_year = law_num_match.group(2)
            if len(law_year) == 2:
                law_year = "20" + law_year
        else:
            # Fallback for preamble/table of contents before the first law header
            law_number = "Unknown"
            law_year = "2022"

        # 2. Split individual articles within the matched law
        article_pattern = r"(مادة\s*\(?\s*\d+\s*\)?\b[\s\S]*?)(?=(?:مادة\s*\(?\s*\d+\s*\)?\b)|$)"
        articles = re.findall(article_pattern, law_text, flags=re.MULTILINE)

        for art_text in articles:
            art_text_cleaned = art_text.strip()
            if not art_text_cleaned:
                continue

            art_num_match = re.search(r"مادة\s*\(?\s*(\d+)\s*\)?", art_text_cleaned)
            article_number = art_num_match.group(1) if art_num_match else "Unknown"

            documents.append(
                Document(
                    page_content=art_text_cleaned,
                    metadata={
                        "law_number": law_number,
                        "law_year": law_year,
                        "article_number": article_number,
                        "source": f"Law{law_number}_Art{article_number}",
                        "char_count": len(art_text_cleaned)
                    }
                )
            )

    return documents

# Helper for standardizing Document IDs
def get_robust_doc_id(doc: Document) -> str:
    law_no = doc.metadata.get("law_number", "Unknown")
    art_no = doc.metadata.get("article_number", "Unknown")
    return f"Law{law_no}_Art{art_no}"

# Re-parse file directly
with open("/kaggle/working/sanitized_gazette_text.txt", "r", encoding="utf-8") as f:
    full_gazette_text = f.read()

docs = parse_libyan_laws_v4(full_gazette_text)
# Filter out non-article metadata headers if present
article_docs = [d for d in docs if d.metadata["article_number"] != "Unknown" and d.metadata["law_number"] != "Unknown"]

doc_id_corpus = [get_robust_doc_id(d) for d in article_docs]

print(f"Total Article Chunks Parsed: {len(article_docs)}")
print("Sample Fixed IDs:", doc_id_corpus[:15])

# Count distinct laws found
distinct_laws = set(d.metadata.get("law_number") for d in article_docs)
print(f"Distinct Law Numbers Detected in Corpus: {distinct_laws}")

Total Article Chunks Parsed: 127
Sample Fixed IDs: ['Law1_Art1', 'Law1_Art2', 'Law1_Art3', 'Law1_Art4', 'Law1_Art5', 'Law1_Art7', 'Law1_Art10', 'Law1_Art11', 'Law1_Art12', 'Law1_Art13', 'Law1_Art14', 'Law1_Art15', 'Law1_Art16', 'Law1_Art17', 'Law1_Art21']
Distinct Law Numbers Detected in Corpus: {'1'}


In [54]:
import re
from typing import List
from langchain_core.documents import Document

def parse_libyan_laws_v5(raw_text: str) -> List[Document]:
    documents = []
    
    # Split text by Law Headers
    law_title_pattern = r"(قانون\s+رقم\s*\(?\s*\d+\s*\)?\s+لسنة\s+\d{2,4}[^\n]*)"
    splits = re.split(law_title_pattern, raw_text)
    
    law_blocks = []
    if len(splits) > 1:
        for i in range(1, len(splits), 2):
            header = splits[i]
            body = splits[i+1] if (i+1) < len(splits) else ""
            
            # FILTER: Only keep blocks that actually contain article definitions (skips preamble citations)
            if "مادة" in body or "المادة" in body:
                law_blocks.append((header, header + body))
    else:
        law_blocks = [("", raw_text)]

    for header, law_text in law_blocks:
        # Extract law number strictly from the valid law block header
        law_num_match = re.search(r"قانون\s+رقم\s*\(?\s*(\d+)\s*\)?\s+لسنة\s+(\d{2,4})", header)
        
        if law_num_match:
            law_number = law_num_match.group(1)
            law_year = law_num_match.group(2)
            if len(law_year) == 2:
                law_year = "20" + law_year
        else:
            law_number = "Unknown"
            law_year = "2022"

        # Split individual articles within the matched law block
        article_pattern = r"(مادة\s*\(?\s*\d+\s*\)?\b[\s\S]*?)(?=(?:مادة\s*\(?\s*\d+\s*\)?\b)|$)"
        articles = re.findall(article_pattern, law_text, flags=re.MULTILINE)

        for art_text in articles:
            art_text_cleaned = art_text.strip()
            if not art_text_cleaned:
                continue

            art_num_match = re.search(r"مادة\s*\(?\s*(\d+)\s*\)?", art_text_cleaned)
            article_number = art_num_match.group(1) if art_num_match else "Unknown"

            documents.append(
                Document(
                    page_content=art_text_cleaned,
                    metadata={
                        "law_number": law_number,
                        "law_year": law_year,
                        "article_number": article_number,
                        "source": f"Law{law_number}_Art{article_number}",
                        "char_count": len(art_text_cleaned)
                    }
                )
            )

    return documents

# Re-run with the updated function
with open("/kaggle/working/sanitized_gazette_text.txt", "r", encoding="utf-8") as f:
    full_gazette_text = f.read()

docs = parse_libyan_laws_v5(full_gazette_text)
article_docs = [d for d in docs if d.metadata["article_number"] != "Unknown" and d.metadata["law_number"] != "Unknown"]

distinct_laws = set(d.metadata.get("law_number") for d in article_docs)
print(f"Total Article Chunks Parsed: {len(article_docs)}")
print(f"Distinct Law Numbers Detected in Corpus: {distinct_laws}")

Total Article Chunks Parsed: 127
Distinct Law Numbers Detected in Corpus: {'1'}


In [56]:
import re
from typing import List
from langchain_core.documents import Document

def parse_libyan_laws_v6(raw_text: str) -> List[Document]:
    documents = []
    
    # 1. Split text globally by law titles
    law_title_pattern = r"(قانون\s+رقم\s*\(?\s*\d+\s*\)?\s+لسنة\s+\d{2,4})"
    splits = re.split(law_title_pattern, raw_text)
    
    law_blocks = []
    if len(splits) > 1:
        # splits[0] is the Table of Contents / Preamble before the first law. Discard it entirely!
        for i in range(1, len(splits), 2):
            header = splits[i]
            body = splits[i+1] if (i+1) < len(splits) else ""
            
            # 2. STRICT VALIDATION: A real law enactment body MUST contain the start of actual articles 
            # (e.g., "مادة (1)" or "المادة الأولى") near its beginning, filtering out TOC index lines.
            has_real_articles = bool(re.search(r"مادة\s*\(?\s*1\s*\)?|المادة\s+الأولى", body))
            
            if has_real_articles:
                law_blocks.append((header, body))
    else:
        law_blocks = [("", raw_text)]

    for header, body in law_blocks:
        # Extract law number strictly from the validated law block header
        law_num_match = re.search(r"قانون\s+رقم\s*\(?\s*(\d+)\s*\)?\s+لسنة\s+(\d{2,4})", header)
        
        if not law_num_match:
            continue
            
        law_number = law_num_match.group(1)
        law_year = law_num_match.group(2)
        if len(law_year) == 2:
            law_year = "20" + law_year

        # 3. Split individual articles within the validated law body
        article_pattern = r"(مادة\s*\(?\s*\d+\s*\)?\b[\s\S]*?)(?=(?:مادة\s*\(?\s*\d+\s*\)?\b)|$)"
        articles = re.findall(article_pattern, body, flags=re.MULTILINE)

        for art_text in articles:
            art_text_cleaned = art_text.strip()
            if not art_text_cleaned:
                continue

            art_num_match = re.search(r"مادة\s*\(?\s*(\d+)\s*\)?", art_text_cleaned)
            if not art_num_match:
                continue
                
            article_number = art_num_match.group(1)

            documents.append(
                Document(
                    page_content=art_text_cleaned,
                    metadata={
                        "law_number": law_number,
                        "law_year": law_year,
                        "article_number": article_number,
                        "source": f"Law{law_number}_Art{article_number}",
                        "char_count": len(art_text_cleaned)
                    }
                )
            )

    return documents

# Execute execution pipeline
with open("/kaggle/working/sanitized_gazette_text.txt", "r", encoding="utf-8") as f:
    full_gazette_text = f.read()

article_docs = parse_libyan_laws_v6(full_gazette_text)

distinct_laws = set(d.metadata.get("law_number") for d in article_docs)
print(f"Total Article Chunks Parsed: {len(article_docs)}")
print(f"Distinct Law Numbers Detected in Corpus: {distinct_laws}")
print("Sample Fixed IDs:", [d.metadata["source"] for d in article_docs[:10]])

Total Article Chunks Parsed: 127
Distinct Law Numbers Detected in Corpus: {'1'}
Sample Fixed IDs: ['Law1_Art1', 'Law1_Art2', 'Law1_Art3', 'Law1_Art4', 'Law1_Art5', 'Law1_Art7', 'Law1_Art10', 'Law1_Art11', 'Law1_Art12', 'Law1_Art13']


In [59]:
# ===================== CELL 1: load text + chunk the FULL gazette =====================
import re, json, numpy as np, pandas as pd
from collections import defaultdict
from langchain_core.documents import Document

TEXT_PATH = "/kaggle/working/sanitized_gazette_text.txt"
full_gazette_text = open(TEXT_PATH, encoding="utf-8").read()

# The gazette contains 3 laws in this order. OCR lost the number of the first one
# (5/2022 = Cybercrime law) -- verify against the PDF if unsure.
LAW_REGISTRY = [
    {"number": "5",  "year": "2022", "title": "مكافحة الجرائم الإلكترونية"},
    {"number": "6",  "year": "2022", "title": "المعاملات الإلكترونية"},
    {"number": "10", "year": "2022", "title": "تنظيم الجريدة الرسمية"},
]
MAX_JUMP = 3                                   # tolerate a few headers lost by OCR
ART_RE = re.compile(r"مادة\s*[()\s]*(\d{1,3})\s*[()]?")   # handles ") 6(" and "(6)"

def parse_gazette(text):
    """New law starts when article numbering restarts at 1. Cross-references such as
    'ال مادة (37) من هذا القانون' are rejected because they don't continue the sequence."""
    text = re.sub(r"\s+", " ", text)
    marks, law_idx, last = [], -1, 0
    for m in ART_RE.finditer(text):
        n = int(m.group(1))
        if text[max(0, m.start() - 6):m.start()].strip().endswith("ال"):
            continue
        if n == 1 and (law_idx == -1 or last > 1):
            law_idx += 1; last = 1
        elif law_idx >= 0 and last < n <= last + MAX_JUMP:
            last = n
        else:
            continue
        marks.append((m.start(), law_idx, n))
    docs = []
    for i, (s, li, n) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else len(text)
        body = text[s:end].strip()
        law = LAW_REGISTRY[li]
        docs.append(Document(page_content=body, metadata={
            "law_number": law["number"], "law_year": law["year"], "law_title": law["title"],
            "article_number": n, "source": f"Law{law['number']}_Art{n}", "char_count": len(body)}))
    return docs

docs = parse_gazette(full_gazette_text)
doc_ids = [d.metadata["source"] for d in docs]
assert len(doc_ids) == len(set(doc_ids)), "duplicate article ids!"
by = defaultdict(list)
for d in docs: by[d.metadata["law_number"]].append(d.metadata["article_number"])
print(f"Total chunks: {len(docs)}")
for k, v in by.items():
    print(f"  Law {k}: {len(v)} articles, max={max(v)}, missing={sorted(set(range(1, max(v)+1)) - set(v))}")



Total chunks: 142
  Law 5: 53 articles, max=53, missing=[]
  Law 6: 86 articles, max=87, missing=[21]
  Law 10: 3 articles, max=3, missing=[]


In [62]:

# ===================== CELL 2: rebuild EVERY index from the new docs =====================
# --- free GPU memory held by earlier cells (Qwen-7B, old GPU FAISS index + its temp pool, caches) ---
import gc, torch
for _name in ("model", "tokenizer", "vector_db", "res", "gpu_index", "inputs", "outputs", "reranker"):
    globals().pop(_name, None)          # reranker is reloaded below (needs to be on a clean GPU)
gc.collect(); torch.cuda.empty_cache()
print(f"GPU free after cleanup: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")
# (the old run left bm25_index / doc_id_corpus / vector_db pointing at stale chunks)
import torch
from rank_bm25 import BM25Okapi
from langchain_community.vectorstores import FAISS
try:
    embeddings
except NameError:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
        encode_kwargs={"normalize_embeddings": True, "batch_size": 8})
embeddings._client.max_seq_length = 1024   # longest chunk is ~1.5k tokens; default 8192 wastes VRAM
embeddings._client.half()                  # fp16: halves the bge-m3 footprint on the T4
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=1024,
                        device="cuda" if torch.cuda.is_available() else "cpu",
                        model_kwargs={"torch_dtype": torch.float16})
 
# 142 vectors -> a CPU FAISS index is instant and avoids GPU-index (de)serialization errors
with torch.inference_mode():
    vector_db = FAISS.from_documents(docs, embeddings)
torch.cuda.empty_cache()
vector_db.save_local("faiss_libyan_laws_full")
 
def normalize_ar(t):
    t = re.sub(r"[ً-ْـ]", "", t)          # tashkeel + tatweel
    t = re.sub(r"[إأآا]", "ا", t).replace("ى", "ي").replace("ة", "ه")
    t = re.sub(r"[^ء-ي0-9\s]", " ", t)
    return t.split()
STOP = {"ما", "هي", "هل", "في", "من", "علي", "الي", "عن", "او", "ان", "هو", "التي", "الذي", "كل", "ذلك"}
def tok(t): return [w[2:] if w.startswith("ال") and len(w) > 4 else w for w in normalize_ar(t) if w not in STOP]
 
bm25_index = BM25Okapi([tok(d.page_content) for d in docs])
 

GPU free after cleanup: 9.8 GB


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [67]:
# ===================== CELL 3: retrieval strategies =====================
def bm25_retrieve(q, k):
    s = bm25_index.get_scores(tok(q)); return [doc_ids[i] for i in np.argsort(s)[::-1][:k]]
def dense_retrieve(q, k):
    return [d.metadata["source"] for d in vector_db.similarity_search(q, k=k)]
def rerank_retrieve(q, k, candidates=20):
    cand = vector_db.similarity_search(q, k=candidates)
    sc = reranker.predict([[q, d.page_content] for d in cand])
    return [cand[i].metadata["source"] for i in np.argsort(sc)[::-1][:k]]
def hybrid_retrieve(q, k, rrf_k=60):                      # BM25 + dense via reciprocal rank fusion
    score = defaultdict(float)
    for lst in (bm25_retrieve(q, 20), dense_retrieve(q, 20)):
        for r, i in enumerate(lst): score[i] += 1 / (rrf_k + r + 1)
    return [i for i, _ in sorted(score.items(), key=lambda x: -x[1])[:k]]



In [68]:
# ===================== CELL 4: benchmark (ground truth checked against the text) =====================
bench = [
 ("ما هي عقوبة الدخول غير المشروع أو اختراق نظام معلوماتي؟",            ["Law5_Art12", "Law5_Art11"]),
 ("ما عقوبة الهاكرز والقرصنة واختراق موقع إلكتروني؟",                    ["Law5_Art12", "Law5_Art11"]),
 ("ما هي تعريفات المصطلحات الواردة في قانون الجرائم الإلكترونية؟",       ["Law5_Art1"]),
 ("ما عقوبة الاستيلاء على أدوات التعريف والهوية الرقمية لشخص آخر؟",      ["Law5_Art18"]),
 ("ما عقوبة تقليد البطاقة المصرفية الإلكترونية؟",                         ["Law5_Art28"]),
 ("ما حكم حيازة وسائل التشفير دون ترخيص؟",                               ["Law5_Art39", "Law5_Art9"]),
 ("ما عقوبة التنصت غير المشروع على الاتصالات؟",                          ["Law5_Art47"]),
 ("هل يسأل الشخص المعنوي جنائيا عن الجرائم الإلكترونية؟",                ["Law5_Art48"]),
 ("هل للتوقيع الإلكتروني نفس الأثر القانوني للتوقيع التقليدي؟",           ["Law6_Art13", "Law6_Art10"]),
 ("ما التزامات صاحب التوقيع الإلكتروني؟",                                 ["Law6_Art11"]),
 ("متى يعلق مقدم خدمات التصديق شهادة المصادقة الإلكترونية؟",              ["Law6_Art39"]),
 ("هل يحق للمستهلك إرجاع المنتج في العقود الإلكترونية وخلال كم يوم؟",     ["Law6_Art52"]),
 ("من يتحمل مسؤولية فقدان أو سرقة البطاقة المصرفية؟",                     ["Law6_Art69", "Law6_Art68"]),
 ("ما المعلومات التي يجب على البائع توفيرها للمستهلك قبل إبرام العقد؟",   ["Law6_Art48"]),
 ("ما الذي عدلته المادة الأولى من قانون تنظيم الجريدة الرسمية؟",          ["Law10_Art1"]),
]
missing = [g for _, gt in bench for g in gt if g not in set(doc_ids)]
assert not missing, f"ground-truth ids not in corpus: {missing}"



In [69]:
# ===================== CELL 5: run evaluation =====================
STRATS = {"BM25": bm25_retrieve, "Dense (bge-m3)": dense_retrieve,
          "Hybrid (RRF)": hybrid_retrieve, "Dense + Reranker": rerank_retrieve}

def metrics(ret, gt, k):
    top = ret[:k]; hits = set(top) & set(gt)
    rr = next((1 / (i + 1) for i, d in enumerate(top) if d in gt), 0.0)
    return len(hits) / k, len(hits) / len(gt), float(bool(hits)), rr

rows, per_query = [], []
for k in (1, 3, 5):
    for name, fn in STRATS.items():
        m = np.array([metrics(fn(q, k), gt, k) for q, gt in bench])
        rows.append({"K": k, "Strategy": name, "Precision@K": m[:, 0].mean(),
                     "Recall@K": m[:, 1].mean(), "Hit@K": m[:, 2].mean(), "MRR": m[:, 3].mean()})
eval_df = pd.DataFrame(rows).round(3)
print(f"Evaluated on {len(docs)} chunks (3 laws, full gazette), {len(bench)} queries\n")
print(eval_df.to_string(index=False))
eval_df.to_csv("/kaggle/working/rag_eval_results.csv", index=False)

# per-query view @5 so you can see exactly which questions fail
for q, gt in bench:
    per_query.append({"query": q[:45], "gt": ",".join(gt),
                      **{n: (next((i + 1 for i, d in enumerate(fn(q, 5)) if d in gt), None)) for n, fn in STRATS.items()}})
print("\nRank of first relevant article (None = not in top 5):")
print(pd.DataFrame(per_query).to_string(index=False))

Evaluated on 142 chunks (3 laws, full gazette), 15 queries

 K         Strategy  Precision@K  Recall@K  Hit@K   MRR
 1             BM25        0.667     0.500  0.667 0.667
 1   Dense (bge-m3)        0.800     0.733  0.800 0.800
 1     Hybrid (RRF)        0.733     0.600  0.733 0.733
 1 Dense + Reranker        0.800     0.700  0.800 0.800
 3             BM25        0.311     0.700  0.800 0.733
 3   Dense (bge-m3)        0.378     0.900  0.933 0.856
 3     Hybrid (RRF)        0.400     0.933  1.000 0.856
 3 Dense + Reranker        0.378     0.900  1.000 0.889
 5             BM25        0.213     0.800  0.867 0.750
 5   Dense (bge-m3)        0.227     0.900  0.933 0.856
 5     Hybrid (RRF)        0.253     0.967  1.000 0.856
 5 Dense + Reranker        0.253     0.967  1.000 0.889

Rank of first relevant article (None = not in top 5):
                                        query                    gt  BM25  Dense (bge-m3)  Hybrid (RRF)  Dense + Reranker
ما هي عقوبة الدخول غير المشروع أو ا

In [70]:
# ===================== CELL 6: ask a question and inspect everything =====================
# Needs CELL 1-3 to have run (docs, vector_db, bm25_index, reranker, bm25_retrieve, dense_retrieve).
import gc, textwrap, torch, numpy as np
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- load the LLM once (4-bit Qwen2.5-7B, ~5.5 GB). Skipped if already loaded. ----
LLM_ID = "Qwen/Qwen2.5-7B-Instruct"
if "llm" not in globals():
    gc.collect(); torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_ID, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16))
    print("LLM loaded.")

by_id = {d.metadata["source"]: d for d in docs}

def retrieve(q, top_n=3, pool=15):
    """hybrid (BM25 + dense, RRF) -> cross-encoder rerank. Returns [(doc, rerank_score, bm25_rank, dense_rank)]."""
    bm, de = bm25_retrieve(q, pool), dense_retrieve(q, pool)
    fused = defaultdict(float)
    for lst in (bm, de):
        for r, i in enumerate(lst): fused[i] += 1 / (60 + r + 1)
    cand = [by_id[i] for i, _ in sorted(fused.items(), key=lambda x: -x[1])[:pool]]
    sc = reranker.predict([[q, d.page_content] for d in cand], batch_size=4)
    out = []
    for j in np.argsort(sc)[::-1][:top_n]:
        i = cand[j].metadata["source"]
        out.append((cand[j], float(sc[j]),
                    bm.index(i) + 1 if i in bm else None, de.index(i) + 1 if i in de else None))
    return out

SYSTEM = ("أنت مستشار قانوني متخصص في التشريعات الليبية. أجب باللغة العربية بالاعتماد فقط على النصوص "
          "القانونية المرفقة. اذكر رقم القانون وسنته ورقم المادة عند الاستشهاد. النصوص مستخرجة بالتعرف الضوئي "
          "وقد تحتوي على أخطاء إملائية؛ افهم المقصود منها. إذا لم تجد الإجابة في النصوص فقل: "
          "'لا تتوفر معلومات كافية في النصوص المتاحة.' ولا تختلق أحكاما.")

def ask(q, top_n=3, generate=True, show_prompt=False, max_new_tokens=400):
    hits = retrieve(q, top_n)
    print("=" * 90, f"\nالسؤال: {q}\n" + "=" * 90)
    print("\n## RETRIEVED (after hybrid + rerank)")
    ctx = ""
    for n, (d, s, br, dr) in enumerate(hits, 1):
        m = d.metadata
        print(f"\n[{n}] قانون {m['law_number']} لسنة {m['law_year']} ({m['law_title']}) - مادة ({m['article_number']})"
              f"   rerank={s:.2f} | bm25 rank={br} | dense rank={dr}")
        print(textwrap.shorten(d.page_content, 600, placeholder=" ..."))
        ctx += f"\n--- المصدر [{n}]: قانون رقم ({m['law_number']}) لسنة {m['law_year']} - مادة ({m['article_number']}) ---\n{d.page_content}\n"
    if not generate:
        return hits
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"النصوص القانونية:\n{ctx}\n\nالسؤال: {q}"}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    if show_prompt: print("\n## PROMPT\n", prompt)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.inference_mode():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    ans = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print("\n## ANSWER\n" + ans)
    del inputs, out; torch.cuda.empty_cache()
    return hits

# ---- interactive loop: type a question, press Enter. Empty line or 'q' quits. ----
# Prefix with '?' to see retrieval only (no LLM), e.g.  ?ما عقوبة التنصت
while True:
    q = input("\nاسأل (Enter فارغ للخروج): ").strip()
    if q.lower() in ("", "q", "quit", "exit"): break
    if q.startswith("?"): ask(q[1:].strip(), generate=False)
    else:                 ask(q)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

LLM loaded.



اسأل (Enter فارغ للخروج):  ما هي العقوبة في حال اختراق حسابات اشخاص اخرين على الانترنت


السؤال: ما هي العقوبة في حال اختراق حسابات اشخاص اخرين على الانترنت

## RETRIEVED (after hybrid + rerank)

[1] قانون 5 لسنة 2022 (مكافحة الجرائم الإلكترونية) - مادة (47)   rerank=0.17 | bm25 rank=11 | dense rank=3
مادة (47) التصت غير الشروع سنة كل من تصت يعاقب بالحس مدة لا تقل لصالح شبكة المعلومات الدولية غيره على الاتصالات تجرى لصالح التي عبر أو اي وسيلة الكترونية دقم الصفمة:19 العدد ) 1( وتكون العقوبة السجن إذاكان التصت بقصد الحصول اسر على ار حكومية او امنية او عسكرية أو مصرفية . فإذا نشر الأسرار المذكورة بالفقرة السابقة شبكة المعلومات الدولية عبل شخصا او جهةً اوأي وسيلة إلكترونية مكن أخرى اخرى الحصول عليها تكون العقوبة السجن المؤبد .

[2] قانون 5 لسنة 2022 (مكافحة الجرائم الإلكترونية) - مادة (13)   rerank=0.12 | bm25 rank=None | dense rank=4
مادة (13) الاعتراض أو التعرض يعاقب بسالحس مدةلاتقلعنسنة وبغرامة لاتقلعن (1000) ألف دينار ولاتزيد (5000) خمسة آلاف عل على بيانات رقمية أو للربط مع أنظمة الكترونية اخرى .

[3] قانون 5 لسنة 2022 (مكافحة الجرائم الإلكترونية) - مادة (31)   rerank=0.0


اسأل (Enter فارغ للخروج):  كيف ترقيم صفحات الجريدة


السؤال: كيف ترقيم صفحات الجريدة

## RETRIEVED (after hybrid + rerank)

[1] قانون 10 لسنة 2022 (تنظيم الجريدة الرسمية) - مادة (1)   rerank=0.70 | bm25 rank=1 | dense rank=1
مادة (1) تعسدل المسادة (3) من القسانون رقم (8) لسسذة 2011م، بشسان سمية بحيث يكون نصها الجريدة الر تنظيم كالآتي: 1- ترسسل اد المعحدة المسو للنشسر بمجلس الذسواب ويسم تسدوينها تاريخ ورودها . الجريسدة الرسمية بسامر من رذسيس مجلحس 2 - يم النشر _اب، أون يكلف بذلك من نوابه. النو 3 - يم اد-دى طبسع إدارة بمجلس النواب . رقم المفمة:57 العدد ) 1( 4-ترقم صفحات برقم مسلسل متواصل من بداية السنة اعلسى صفحة آخرهاء ويذكر حتسى فسي الرقم المسلسل ورقم عدد الجريدة . التسرقيم المذكور اعتبازان تاريخ العمل بهذا يبسدا -5 القانون . 6-يم حصر جميع التشريعات والقسرارات يجب التي نشرها والتسي السابق منسذ عسام 2011م تنشسر ...

[2] قانون 6 لسنة 2022 (المعاملات الإلكترونية) - مادة (87)   rerank=0.00 | bm25 rank=4 | dense rank=None
مادة (87) مل بأحكام هذا القانون من تاريخ صدوره ويلغى كل حكم يخالفه وينشر في ووسائل الإعلام. مجلس النواب صدرفى مدينة طبرق .


اسأل (Enter فارغ للخروج):  
